In [0]:
import requests
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    TimestampType
)

Configuration

In [0]:
API_KEY = "97RL93JHB1AMKMQG"

TICKERS = [
    "IBM",
    "AAPL",
    "MSFT"
]

CATALOG = "workspace"
SCHEMA = "stock_project"

RAW_PRICE_TABLE = f"{CATALOG}.{SCHEMA}.raw_stock_prices"
RAW_COMPANY_TABLE = f"{CATALOG}.{SCHEMA}.raw_company_info"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

Function for historical stock prices

In [0]:
def get_daily_stock_prices(ticker):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "outputsize": "compact",
        "apikey": API_KEY
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    if "Note" in data:
        raise RuntimeError(
            f"Alpha Vantage rate limit reached: {data['Note']}"
        )

    if "Information" in data:
        raise RuntimeError(
            f"Alpha Vantage message: {data['Information']}"
        )

    if "Error Message" in data:
        raise RuntimeError(
            f"Invalid API request: {data['Error Message']}"
        )

    time_series = data.get("Time Series (Daily)")

    if not time_series:
        raise RuntimeError(
            f"No daily price information returned for {ticker}"
        )

    rows = []

    for trade_date, values in time_series.items():
        rows.append({
            "ticker": ticker,
            "trade_date": trade_date,
            "open": float(values["1. open"]),
            "high": float(values["2. high"]),
            "low": float(values["3. low"]),
            "close": float(values["4. close"]),
            "volume": int(values["5. volume"]),
            "ingested_at": datetime.utcnow()
        })

    return rows

Function for company information

In [0]:
def get_company_info(ticker):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "OVERVIEW",
        "symbol": ticker,
        "apikey": API_KEY
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    if "Note" in data:
        raise RuntimeError(
            f"Alpha Vantage rate limit reached: {data['Note']}"
        )

    if "Information" in data:
        raise RuntimeError(
            f"Alpha Vantage message: {data['Information']}"
        )

    if "Error Message" in data:
        raise RuntimeError(
            f"Invalid API request: {data['Error Message']}"
        )

    if not data or "Symbol" not in data:
        raise RuntimeError(
            f"No company information returned for {ticker}"
        )

    return {
        "ticker": ticker,
        "company_name": data.get("Name"),
        "description": data.get("Description"),
        "exchange": data.get("Exchange"),
        "currency": data.get("Currency"),
        "country": data.get("Country"),
        "sector": data.get("Sector"),
        "industry": data.get("Industry"),
        "market_capitalization": data.get("MarketCapitalization"),
        "pe_ratio": data.get("PERatio"),
        "dividend_yield": data.get("DividendYield"),
        "week_52_high": data.get("52WeekHigh"),
        "week_52_low": data.get("52WeekLow"),
        "ingested_at": datetime.utcnow()
    }

Retrieve the data

In [0]:
all_price_rows = []
all_company_rows = []

for ticker in TICKERS:
    try:
        print(f"Retrieving daily prices for {ticker}")

        price_rows = get_daily_stock_prices(ticker)
        all_price_rows.extend(price_rows)

        time.sleep(13)

        print(f"Retrieving company information for {ticker}")

        company_row = get_company_info(ticker)
        all_company_rows.append(company_row)

        time.sleep(13)

    except Exception as error:
        print(f"Failed to retrieve {ticker}: {error}")

Retrieving daily prices for IBM


/home/spark-ca25eb23-bbfd-4be1-bde9-61/.ipykernel/328/command-7359151947291537-1612088837:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow()


Retrieving company information for IBM


/home/spark-ca25eb23-bbfd-4be1-bde9-61/.ipykernel/328/command-7359151947291540-356092266:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow()


Retrieving daily prices for AAPL


...

Retrieving company information for AAPL


/home/spark-ca25eb23-bbfd-4be1-bde9-61/.ipykernel/328/command-7359151947291540-356092266:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow()


Retrieving daily prices for MSFT


/home/spark-ca25eb23-bbfd-4be1-bde9-61/.ipykernel/328/command-7359151947291537-1612088837:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow()


Retrieving company information for MSFT


Save price data

In [0]:
price_schema = StructType([
    StructField("ticker", StringType(), False),
    StructField("trade_date", StringType(), False),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("ingested_at", TimestampType(), True)
])

if all_price_rows:
    price_df = spark.createDataFrame(
        all_price_rows,
        schema=price_schema
    )

    price_df = (
        price_df
        .withColumn(
            "trade_date",
            F.to_date("trade_date")
        )
        .dropDuplicates([
            "ticker",
            "trade_date"
        ])
    )

    price_df.createOrReplaceTempView("new_stock_prices")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RAW_PRICE_TABLE}
USING DELTA
AS
SELECT *
FROM new_stock_prices
WHERE 1 = 0
""")

spark.sql(f"""
MERGE INTO {RAW_PRICE_TABLE} AS target
USING new_stock_prices AS source
ON target.ticker = source.ticker
AND target.trade_date = source.trade_date

WHEN MATCHED THEN UPDATE SET
    target.open = source.open,
    target.high = source.high,
    target.low = source.low,
    target.close = source.close,
    target.volume = source.volume,
    target.ingested_at = source.ingested_at

WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Save company data

In [0]:
company_schema = StructType([
    StructField("ticker", StringType(), False),
    StructField("company_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("exchange", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("country", StringType(), True),
    StructField("sector", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("market_capitalization", StringType(), True),
    StructField("pe_ratio", StringType(), True),
    StructField("dividend_yield", StringType(), True),
    StructField("week_52_high", StringType(), True),
    StructField("week_52_low", StringType(), True),
    StructField("ingested_at", TimestampType(), True)
])

if all_company_rows:
    company_df = spark.createDataFrame(
        all_company_rows,
        schema=company_schema
    )

    company_df.createOrReplaceTempView("new_company_info")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RAW_COMPANY_TABLE}
USING DELTA
AS
SELECT *
FROM new_company_info
WHERE 1 = 0
""")

spark.sql(f"""
MERGE INTO {RAW_COMPANY_TABLE} AS target
USING new_company_info AS source
ON target.ticker = source.ticker

WHEN MATCHED THEN UPDATE SET
    target.company_name = source.company_name,
    target.description = source.description,
    target.exchange = source.exchange,
    target.currency = source.currency,
    target.country = source.country,
    target.sector = source.sector,
    target.industry = source.industry,
    target.market_capitalization = source.market_capitalization,
    target.pe_ratio = source.pe_ratio,
    target.dividend_yield = source.dividend_yield,
    target.week_52_high = source.week_52_high,
    target.week_52_low = source.week_52_low,
    target.ingested_at = source.ingested_at

WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Verify the ingestion

In [0]:
display(
    spark.table(RAW_PRICE_TABLE)
    .orderBy(
        F.col("ticker"),
        F.col("trade_date").desc()
    )
)

ticker,trade_date,open,high,low,close,volume,ingested_at
AAPL,2026-07-09,310.51,316.53,308.16,316.22,48124490,2026-07-10T13:26:58.623Z
AAPL,2026-07-08,311.91,314.82,307.05,313.39,41323480,2026-07-10T13:26:58.623Z
AAPL,2026-07-07,315.29,315.48,310.15,310.66,42490002,2026-07-10T13:26:58.623Z
AAPL,2026-07-06,307.36,314.2,307.0,312.66,53589977,2026-07-10T13:26:58.623Z
AAPL,2026-07-02,294.12,309.42,293.68,308.63,75400626,2026-07-10T13:26:58.623Z
AAPL,2026-07-01,293.44,296.59,289.195,294.38,50164232,2026-07-10T13:26:58.623Z
AAPL,2026-06-30,281.17,289.94,280.695,289.36,65100155,2026-07-10T13:26:58.623Z
AAPL,2026-06-29,286.73,288.3697,279.85,281.74,66427002,2026-07-10T13:26:58.623Z
AAPL,2026-06-26,275.0,285.95,274.21,283.78,261775450,2026-07-10T13:26:58.623Z
AAPL,2026-06-25,287.4,288.8,273.75,275.15,107253659,2026-07-10T13:26:58.623Z


In [0]:
display(
    spark.table(RAW_COMPANY_TABLE)
)

ticker,company_name,description,exchange,currency,country,sector,industry,market_capitalization,pe_ratio,dividend_yield,week_52_high,week_52_low,ingested_at
IBM,International Business Machines,"International Business Machines Corporation (IBM) is an American multinational technology company headquartered in Armonk, New York, with operations in over 170 countries. The company began in 1911, founded in Endicott, New York, as the Computing-Tabulating-Recording Company (CTR) and was renamed International Business Machines in 1924. IBM is incorporated in New York. IBM produces and sells computer hardware, middleware and software, and provides hosting and consulting services in areas ranging from mainframe computers to nanotechnology. IBM is also a major research organization, holding the record for most annual U.S. patents generated by a business (as of 2020) for 28 consecutive years. Inventions by IBM include the automated teller machine (ATM), the floppy disk, the hard disk drive, the magnetic stripe card, the relational database, the SQL programming language, the UPC barcode, and dynamic random-access memory (DRAM). The IBM mainframe, exemplified by the System/360, was the dominant computing platform during the 1960s and 1970s.",NYSE,USD,USA,TECHNOLOGY,INFORMATION TECHNOLOGY SERVICES,277548106000,26.75,0.0222,332.46,212.34,2026-07-10T13:26:45.563Z
MSFT,Microsoft Corporation,"Microsoft Corporation is an American multinational technology company which produces computer software, consumer electronics, personal computers, and related services. Its best known software products are the Microsoft Windows line of operating systems, the Microsoft Office suite, and the Internet Explorer and Edge web browsers. Its flagship hardware products are the Xbox video game consoles and the Microsoft Surface lineup of touchscreen personal computers. Microsoft ranked No. 21 in the 2020 Fortune 500 rankings of the largest United States corporations by total revenue; it was the world's largest software maker by revenue as of 2016. It is considered one of the Big Five companies in the U.S. information technology industry, along with Google, Apple, Amazon, and Facebook.",NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,2855193018000,22.84,0.0093,551.05,349.2,2026-07-10T13:27:38.221Z
AAPL,Apple Inc.,"Apple Inc. is an American multinational technology company that specializes in consumer electronics, computer software, and online services. Apple is the world's largest technology company by revenue (totalling $274.5 billion in 2020) and, since January 2021, the world's most valuable company. As of 2021, Apple is the world's fourth-largest PC vendor by unit sales, and fourth-largest smartphone manufacturer. It is one of the Big Five American information technology companies, along with Amazon, Google, Microsoft, and Facebook.",NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,4644435657000,38.24,0.0033,317.4,200.7,2026-07-10T13:27:12.069Z
